# Classificador de Grãos de Soja — Treino EfficientNet-B0

**Dataset:** SoyaBeans Classifications (Roboflow / hansaka-sudusinghe) — MIT License  
**Modelo:** EfficientNet-B0 com Transfer Learning (ImageNet → 5 classes)  
**Framework:** TensorFlow / Keras  

### Pré-requisito
1. Faça upload do dataset extraído no Google Drive na raiz ("Meu Drive")  
   O caminho esperado é: `Meu Drive/SoyaBeans Classifications.v2i.folder (Unzipped Files)/`  
2. Ative o ambiente de execução com GPU: `Ambiente de execução → Alterar tipo → T4 GPU`

### Classes (5)
| Índice | Pasta no dataset | Rótulo curto |
|--------|------------------|--------------|
| 0 | Broken soybeans | broken |
| 1 | Immature soybeans | immature |
| 2 | Intact soybeans | intact |
| 3 | Skin-damaged soybeans | skin-damaged |
| 4 | Spotted soybeans | spotted |

In [ ]:
# Célula 1 — Verificar GPU e importar bibliotecas
import os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import classification_report, confusion_matrix

# Reprodutibilidade: fixa python random + numpy + tensorflow de uma vez
tf.keras.utils.set_random_seed(42)

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs disponíveis: {gpus}')
assert len(gpus) > 0, 'Sem GPU! Vá em Ambiente de execução → Alterar tipo → T4'

In [ ]:
# Célula 2 — Montar Google Drive e configurar caminhos
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)'
TRAIN_DIR    = f'{DATASET_ROOT}/train'
VALID_DIR    = f'{DATASET_ROOT}/valid'
TEST_DIR     = f'{DATASET_ROOT}/test'
SAVE_DIR     = '/content/drive/MyDrive'

# Verificar se o dataset existe
for path in [TRAIN_DIR, VALID_DIR, TEST_DIR]:
    assert os.path.isdir(path), f'Caminho não encontrado: {path}'
    print(f'OK: {path}')

# Classes exatas (ordem alfabética = índice do modelo)
# O folder "Part of the original soybean images" é ignorado via class_names explícito
CLASS_NAMES = [
    'Broken soybeans',
    'Immature soybeans',
    'Intact soybeans',
    'Skin-damaged soybeans',
    'Spotted soybeans',
]

SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
print('\nClasses:', CLASS_NAMES)

In [ ]:
# Célula 3 — Carregar datasets
IMG_SIZE   = (224, 224)   # Input nativo do EfficientNet-B0
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE
SEED       = 42

def make_dataset(directory, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        class_names=CLASS_NAMES,   # ignora "Part of the original soybean images"
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        shuffle=shuffle,
        seed=SEED,
    )

train_ds = make_dataset(TRAIN_DIR, shuffle=True)
val_ds   = make_dataset(VALID_DIR, shuffle=False)
test_ds  = make_dataset(TEST_DIR,  shuffle=False)

n_train = sum(len(y) for _, y in train_ds)
n_val   = sum(len(y) for _, y in val_ds)
n_test  = sum(len(y) for _, y in test_ds)
print(f'Treino:    {n_train:>6} imagens')
print(f'Validação: {n_val:>6} imagens')
print(f'Teste:     {n_test:>6} imagens')
print(f'Total:     {n_train+n_val+n_test:>6} imagens')

# Cache em DISCO local do Colab (não em RAM) — evita estourar memória com
# 12 mil imagens e acelera as épocas seguintes (ler do Drive a cada época é lento).
train_ds = train_ds.cache('/content/train_cache').prefetch(AUTOTUNE)
val_ds   = val_ds.cache('/content/val_cache').prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [ ]:
# Célula 4 — Visualizar amostras do dataset
plt.figure(figsize=(15, 6))
for images, labels in train_ds.take(1):
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        cls = CLASS_NAMES[np.argmax(labels[i])]
        plt.title(SHORT[cls], fontsize=9)
        plt.axis('off')
plt.suptitle('Amostras do conjunto de treino', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Célula 5 — Data augmentation (aplicado apenas no treino)
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.08),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

train_ds_aug = train_ds.map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE,
)

# Visualizar efeito do augmentation
plt.figure(figsize=(15, 4))
for images, _ in train_ds.take(1):
    sample = images[:1]
    plt.subplot(1, 5, 1)
    plt.imshow(sample[0].numpy().astype('uint8'))
    plt.title('Original', fontsize=9)
    plt.axis('off')
    for i in range(4):
        aug = augmentation(sample, training=True)
        plt.subplot(1, 5, i + 2)
        plt.imshow(aug[0].numpy().astype('uint8'))
        plt.title(f'Aug {i+1}', fontsize=9)
        plt.axis('off')
plt.suptitle('Exemplos de augmentation', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Célula 6 — Construir modelo: Fase 1 (base congelada)
NUM_CLASSES = len(CLASS_NAMES)

base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(*IMG_SIZE, 3),
)
base_model.trainable = False

inputs  = keras.Input(shape=(*IMG_SIZE, 3))
x       = keras.applications.efficientnet.preprocess_input(inputs)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs, name='soja_efficientnet_b0')

total  = sum(np.prod(v.shape) for v in model.trainable_variables)
frozen = sum(np.prod(v.shape) for v in model.non_trainable_variables)
print(f'Parâmetros treináveis:  {total:>10,}')
print(f'Parâmetros congelados:  {frozen:>10,}')

In [ ]:
# Célula 7 — Treino Fase 1 (base congelada, LR alto)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p1 = [
    keras.callbacks.EarlyStopping(
        patience=5, restore_best_weights=True,
        monitor='val_accuracy', verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{SAVE_DIR}/soja_phase1_best.keras',
        save_best_only=True, monitor='val_accuracy', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        patience=3, factor=0.5, monitor='val_loss',
        min_lr=1e-6, verbose=1),
]

# Pesos por classe — compensa o desbalanceamento (spotted tem menos imagens).
# Sem isso, o modelo tende a "ignorar" a classe minoritária.
from sklearn.utils.class_weight import compute_class_weight
y_train = np.concatenate([np.argmax(y, axis=1) for _, y in train_ds])
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(weights)}
print('Pesos por classe:', {SHORT[CLASS_NAMES[i]]: round(w, 2) for i, w in class_weight.items()})

print('=== FASE 1 — Base congelada (só a cabeça treina) ===')
history1 = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks_p1,
    class_weight=class_weight,
)

In [ ]:
# Célula 8 — Curvas de aprendizado
def plot_history(h, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(h.history['accuracy'],     label='Treino',    linewidth=2)
    ax1.plot(h.history['val_accuracy'], label='Validação', linewidth=2)
    ax1.set_title(f'{title} — Acurácia')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Acurácia')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.plot(h.history['loss'],     label='Treino',    linewidth=2)
    ax2.plot(h.history['val_loss'], label='Validação', linewidth=2)
    ax2.set_title(f'{title} — Loss')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_history(history1, 'Fase 1 — Base congelada')

In [ ]:
# Célula 9 — Fase 2: Fine-tuning (descongela as últimas 30 camadas)
# Só execute se a Fase 1 atingiu val_accuracy >= 85%

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
# Manter BatchNorm congelada mesmo na cauda descongelada — descongelar a BN
# destrói as estatísticas (mean/variance) herdadas do ImageNet e piora o resultado.
for layer in base_model.layers[-30:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
print(f'Camadas treináveis na base: {trainable}/{len(base_model.layers)}')

# LR 100x menor que Fase 1 para não destruir os pesos do ImageNet
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p2 = [
    keras.callbacks.EarlyStopping(
        patience=5, restore_best_weights=True,
        monitor='val_accuracy', verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{SAVE_DIR}/soja_phase2_best.keras',
        save_best_only=True, monitor='val_accuracy', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        patience=3, factor=0.5, monitor='val_loss',
        min_lr=1e-7, verbose=1),
]

print('=== FASE 2 — Fine-tuning (últimas 30 camadas) ===')
history2 = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_p2,
    class_weight=class_weight,
)
plot_history(history2, 'Fase 2 — Fine-tuning')

In [ ]:
# Célula 10 — Avaliação no conjunto de TESTE
print('=== Avaliação no conjunto de TESTE (dados nunca vistos) ===')
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Loss: {test_loss:.4f}  |  Acurácia: {test_acc:.2%}')

# Coletar predições
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.concatenate([np.argmax(y, axis=1) for _, y in test_ds])

short_names = [SHORT[c] for c in CLASS_NAMES]

print('\n=== Relatório de Classificação ===')
print(classification_report(y_true, y_pred, target_names=short_names))

In [ ]:
# Célula 11 — Matriz de confusão
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=short_names,
    yticklabels=short_names,
    cmap='Blues', linewidths=0.5,
)
plt.xlabel('Predito', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title(f'Matriz de Confusão — Acurácia: {test_acc:.1%}', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/soja_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Matriz salva no Drive.')

In [ ]:
# Célula 12 — Salvar modelo final + lista de classes
MODEL_PATH   = f'{SAVE_DIR}/soja_model_final.keras'
CLASSES_PATH = f'{SAVE_DIR}/soja_classes.json'

model.save(MODEL_PATH)
with open(CLASSES_PATH, 'w') as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

print(f'Modelo salvo:  {MODEL_PATH}')
print(f'Classes salvas: {CLASSES_PATH}')
print()
print('Próximos passos:')
print('1. Baixe soja_model_final.keras e soja_classes.json do Drive')
print('2. Adicione-os ao repositório do HF Space junto com app.py e requirements.txt')
print('3. Faça push para o Space → a demo estará disponível online')

In [ ]:
# Célula 13 (OPCIONAL) — Inspecionar erros: ver imagens que o modelo errou
wrong_idx = np.where(y_pred != y_true)[0]
print(f'Total de erros: {len(wrong_idx)} de {len(y_true)} ({len(wrong_idx)/len(y_true):.1%})')

# Montar lista de imagens de teste
test_images_list = []
test_labels_list = []
for imgs, lbls in test_ds:
    test_images_list.extend(imgs.numpy().astype('uint8'))
    test_labels_list.extend(np.argmax(lbls.numpy(), axis=1))
test_images_list = np.array(test_images_list)

# Mostrar até 10 erros
show = wrong_idx[:10]
plt.figure(figsize=(15, 4))
for i, idx in enumerate(show):
    plt.subplot(2, 5, i + 1)
    plt.imshow(test_images_list[idx])
    real  = short_names[y_true[idx]]
    pred  = short_names[y_pred[idx]]
    conf  = y_pred_probs[idx][y_pred[idx]]
    plt.title(f'Real: {real}\nPred: {pred} {conf:.0%}', fontsize=7, color='red')
    plt.axis('off')
plt.suptitle('Exemplos de erros do modelo', fontsize=12)
plt.tight_layout()
plt.show()